In [1]:
# ================================
# Setup Cell: S&P 500 Universe + Metadata
# ================================

import os
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---- Project Structure ----
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
UNIVERSE_DIR = DATA_DIR / "universe"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

UNIVERSE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ---- Universe Config ----
UNIVERSE_NAME = "sp500"

TICKER_FILE = UNIVERSE_DIR / f"{UNIVERSE_NAME}.csv"
UNIVERSE_METADATA_FILE = UNIVERSE_DIR / f"{UNIVERSE_NAME}_metadata.csv"

# ---- Always refresh universe metadata from Wikipedia ----
# This captures useful fields:
#   ticker, company, sector, sub_industry, headquarters, date_added, cik, founded
#
# Note:
#   This is current S&P 500 membership metadata, not historical membership.
#   For this project phase, we are using a current-constituent universe.

print("Fetching S&P 500 universe metadata...")

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
table = soup.find("table", {"id": "constituents"})

sp500_raw = pd.read_html(str(table))[0]

sp500_metadata = sp500_raw.rename(columns={
    "Symbol": "ticker",
    "Security": "company",
    "GICS Sector": "sector",
    "GICS Sub-Industry": "sub_industry",
    "Headquarters Location": "headquarters",
    "Date added": "date_added",
    "CIK": "cik",
    "Founded": "founded",
}).copy()

sp500_metadata["ticker"] = (
    sp500_metadata["ticker"]
    .str.replace(".", "-", regex=False)  # BRK.B → BRK-B
    .str.upper()
    .str.strip()
)

sp500_metadata["date_added"] = pd.to_datetime(
    sp500_metadata["date_added"],
    errors="coerce",
)

# Keep clean metadata columns.
metadata_cols = [
    "ticker",
    "company",
    "sector",
    "sub_industry",
    "headquarters",
    "date_added",
    "cik",
    "founded",
]

sp500_metadata = sp500_metadata[
    [c for c in metadata_cols if c in sp500_metadata.columns]
].drop_duplicates(subset=["ticker"])

# ---- Save metadata and ticker-only universe ----
sp500_metadata.to_csv(UNIVERSE_METADATA_FILE, index=False)

tickers_df = sp500_metadata[["ticker"]].copy()
tickers_df.to_csv(TICKER_FILE, index=False)

# ---- Load tickers ----
tickers = (
    tickers_df["ticker"]
    .dropna()
    .str.upper()
    .str.strip()
    .unique()
    .tolist()
)

print(f"\nSaved metadata to: {UNIVERSE_METADATA_FILE}")
print(f"Saved ticker file to: {TICKER_FILE}")
print(f"Loaded {len(tickers)} tickers")
print("Sample:", tickers[:10])

sp500_metadata.head()

Fetching S&P 500 universe metadata...

Saved metadata to: /Users/neilyejjey/stock_signal_engine_v1/data/universe/sp500_metadata.csv
Saved ticker file to: /Users/neilyejjey/stock_signal_engine_v1/data/universe/sp500.csv
Loaded 503 tickers
Sample: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


/var/folders/l3/hj4t10p12gs0m_nkjpjr9kqr0000gp/T/ipykernel_3408/2195897796.py:47: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500_raw = pd.read_html(str(table))[0]


,ticker,company,sector,sub_industry,headquarters,date_added,cik,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [2]:
tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APP',
 'APTV',
 'ACGL',
 'ADM',
 'ARES',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVB',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BRK-B',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'XYZ',
 'BNY',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BF-B',
 'BLDR',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CPT',
 'COF',
 'CAH',
 'CCL',
 'CARR',
 'CVNA',
 'CASY',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW',
 'CHTR',
 'CVX',
 'CMG',
 'CB',
 'CHD',
 'CIEN',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'C',
 'CFG',
 'CLX',
 'CME',
 'CMS',
 'KO',
 'CTSH',
 'COHR',
 'COIN',
 'CL',
 'CMCSA',
 'FIX',
 

In [3]:
# #in case modules not installed
# !pip install --upgrade pip setuptools wheel
# !pip install yfinance

In [4]:
# =========================================
# Download Price Data
# =========================================

import time
import yfinance as yf
import pandas as pd

# ---- Config ----
START_DATE = "2010-01-01"
END_DATE = "2026-01-01"   # exclusive in yfinance
SLEEP_SECONDS = 0.25      # tiny pause to be polite / reduce hiccups

PRICE_FILE = RAW_DIR / "prices.parquet"
FAILED_FILE = RAW_DIR / "failed_tickers.csv"

all_prices = []
failed_tickers = []

for i, ticker in enumerate(tickers, start=1):
    try:
        print(f"[{i}/{len(tickers)}] Downloading {ticker}...")

        df = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=False,   # keep raw OHLC + Adj Close
            actions=True,        # include dividends / splits when available
            progress=False,
            threads=False,
        )

        if df.empty:
            print(f"  -> No data returned for {ticker}")
            failed_tickers.append(ticker)
            continue

        # Flatten columns just in case
        df.columns = [
            col[0] if isinstance(col, tuple) else col
            for col in df.columns
        ]

        # Standardize column names
        df = df.reset_index().rename(columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume",
            "Dividends": "dividends",
            "Stock Splits": "stock_splits",
        })

        # Add ticker
        df["ticker"] = ticker

        # Keep only the columns we care about
        expected_cols = [
            "date",
            "ticker",
            "open",
            "high",
            "low",
            "close",
            "adj_close",
            "volume",
            "dividends",
            "stock_splits",
        ]

        df = df[[col for col in expected_cols if col in df.columns]]

        # Ensure optional columns exist
        if "dividends" not in df.columns:
            df["dividends"] = 0.0

        if "stock_splits" not in df.columns:
            df["stock_splits"] = 0.0

        # Basic cleanup
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values(["ticker", "date"]).drop_duplicates(
            subset=["ticker", "date"]
        )

        all_prices.append(df)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        print(f"  -> Failed for {ticker}: {e}")
        failed_tickers.append(ticker)

# ---- Combine all successful downloads ----
if not all_prices:
    raise ValueError("No price data was downloaded. Check tickers and internet/API availability.")

prices = pd.concat(all_prices, ignore_index=True)
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)

# ---- Save outputs ----
prices.to_parquet(PRICE_FILE, index=False)

failed_df = pd.DataFrame({"ticker": failed_tickers})
failed_df.to_csv(FAILED_FILE, index=False)

print("\nDownload complete.")
print(f"Saved prices to: {PRICE_FILE}")
print(f"Saved failed tickers to: {FAILED_FILE}")
print(f"Price dataset shape: {prices.shape}")
print(f"Unique tickers downloaded: {prices['ticker'].nunique()}")
print(f"Failed tickers: {len(failed_tickers)}")

prices.head()

[1/503] Downloading MMM...
[2/503] Downloading AOS...
[3/503] Downloading ABT...
[4/503] Downloading ABBV...
[5/503] Downloading ACN...
[6/503] Downloading ADBE...
[7/503] Downloading AMD...
[8/503] Downloading AES...
[9/503] Downloading AFL...
[10/503] Downloading A...
[11/503] Downloading APD...
[12/503] Downloading ABNB...
[13/503] Downloading AKAM...
[14/503] Downloading ALB...
[15/503] Downloading ARE...
[16/503] Downloading ALGN...
[17/503] Downloading ALLE...
[18/503] Downloading LNT...
[19/503] Downloading ALL...
[20/503] Downloading GOOGL...
[21/503] Downloading GOOG...
[22/503] Downloading MO...
[23/503] Downloading AMZN...
[24/503] Downloading AMCR...
[25/503] Downloading AEE...
[26/503] Downloading AEP...
[27/503] Downloading AXP...
[28/503] Downloading AIG...
[29/503] Downloading AMT...
[30/503] Downloading AWK...
[31/503] Downloading AMP...
[32/503] Downloading AME...
[33/503] Downloading AMGN...
[34/503] Downloading APH...
[35/503] Downloading ADI...
[36/503] Downloading

$FDXF: possibly delisted; no price data found  (1d 2010-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1767243600")

1 Failed download:
['FDXF']: possibly delisted; no price data found  (1d 2010-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1767243600")


  -> No data returned for FDXF
[197/503] Downloading FIS...
[198/503] Downloading FITB...
[199/503] Downloading FSLR...
[200/503] Downloading FE...
[201/503] Downloading FISV...
[202/503] Downloading FLEX...
[203/503] Downloading F...
[204/503] Downloading FTNT...
[205/503] Downloading FTV...
[206/503] Downloading FOXA...
[207/503] Downloading FOX...
[208/503] Downloading BEN...
[209/503] Downloading FCX...
[210/503] Downloading GRMN...
[211/503] Downloading IT...
[212/503] Downloading GE...
[213/503] Downloading GEHC...
[214/503] Downloading GEV...
[215/503] Downloading GEN...
[216/503] Downloading GNRC...
[217/503] Downloading GD...
[218/503] Downloading GIS...
[219/503] Downloading GM...
[220/503] Downloading GPC...
[221/503] Downloading GILD...
[222/503] Downloading GPN...
[223/503] Downloading GL...
[224/503] Downloading GDDY...
[225/503] Downloading GS...
[226/503] Downloading HAL...
[227/503] Downloading HIG...
[228/503] Downloading HAS...
[229/503] Downloading HCA...
[230/503] 

$HONA: possibly delisted; no price data found  (1d 2010-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1767243600")

1 Failed download:
['HONA']: possibly delisted; no price data found  (1d 2010-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1767243600")


  -> No data returned for HONA
[237/503] Downloading HON...
[238/503] Downloading HRL...
[239/503] Downloading HST...
[240/503] Downloading HWM...
[241/503] Downloading HPQ...
[242/503] Downloading HUBB...
[243/503] Downloading HUM...
[244/503] Downloading HBAN...
[245/503] Downloading HII...
[246/503] Downloading IBM...
[247/503] Downloading IEX...
[248/503] Downloading IDXX...
[249/503] Downloading ITW...
[250/503] Downloading INCY...
[251/503] Downloading IR...
[252/503] Downloading PODD...
[253/503] Downloading INTC...
[254/503] Downloading IBKR...
[255/503] Downloading ICE...
[256/503] Downloading IFF...
[257/503] Downloading IP...
[258/503] Downloading INTU...
[259/503] Downloading ISRG...
[260/503] Downloading IVZ...
[261/503] Downloading INVH...
[262/503] Downloading IQV...
[263/503] Downloading IRM...
[264/503] Downloading JBHT...
[265/503] Downloading JBL...
[266/503] Downloading JKHY...
[267/503] Downloading J...
[268/503] Downloading JNJ...
[269/503] Downloading JCI...
[270

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits
0,2010-01-04,A,22.453505,22.625179,22.267525,22.389128,19.772955,3815561,0.0,0.0
1,2010-01-05,A,22.324751,22.331903,22.002861,22.145924,19.558165,4186031,0.0,0.0
2,2010-01-06,A,22.067240,22.174536,22.002861,22.067240,19.488676,3243779,0.0,0.0
3,2010-01-07,A,22.017166,22.045780,21.816881,22.038628,19.463409,3095172,0.0,0.0
4,2010-01-08,A,21.917025,22.067240,21.745352,22.031473,19.457087,3733918,0.0,0.0


In [5]:
prices

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits
0,2010-01-04,A,22.453505,22.625179,22.267525,22.389128,19.772955,3815561,0.0,0.0
1,2010-01-05,A,22.324751,22.331903,22.002861,22.145924,19.558165,4186031,0.0,0.0
2,2010-01-06,A,22.067240,22.174536,22.002861,22.067240,19.488676,3243779,0.0,0.0
3,2010-01-07,A,22.017166,22.045780,21.816881,22.038628,19.463409,3095172,0.0,0.0
4,2010-01-08,A,21.917025,22.067240,21.745352,22.031473,19.457087,3733918,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
1895351,2025-12-24,ZTS,123.099998,125.690002,123.059998,125.489998,124.415276,2369000,0.0,0.0
1895352,2025-12-26,ZTS,125.160004,126.320000,124.800003,126.230003,125.148941,3226800,0.0,0.0
1895353,2025-12-29,ZTS,126.160004,126.849998,125.559998,125.980003,124.901085,4465600,0.0,0.0
1895354,2025-12-30,ZTS,125.550003,127.599998,125.449997,126.410004,125.327400,3230200,0.0,0.0


In [6]:
# =========================================
# Download Benchmark / ETF Price Data
# =========================================
#
# These benchmarks can be used downstream for:
#   - forward excess return labels
#   - strategy comparison
#   - VOO/SPY benchmark analysis
#
# SPY has longer trading history.
# VOO is closer to the user's desired benchmark but starts in 2010.

BENCHMARK_TICKERS = ["SPY", "VOO"]
BENCHMARK_PRICE_FILE = RAW_DIR / "benchmark_prices.parquet"

benchmark_prices = []
failed_benchmarks = []

for ticker in BENCHMARK_TICKERS:
    try:
        print(f"Downloading benchmark {ticker}...")

        df = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=False,
            actions=True,
            progress=False,
            threads=False,
        )

        if df.empty:
            print(f"  -> No data returned for {ticker}")
            failed_benchmarks.append(ticker)
            continue

        df.columns = [
            col[0] if isinstance(col, tuple) else col
            for col in df.columns
        ]

        df = df.reset_index().rename(columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume",
            "Dividends": "dividends",
            "Stock Splits": "stock_splits",
        })

        df["ticker"] = ticker

        expected_cols = [
            "date",
            "ticker",
            "open",
            "high",
            "low",
            "close",
            "adj_close",
            "volume",
            "dividends",
            "stock_splits",
        ]

        df = df[[col for col in expected_cols if col in df.columns]]

        if "dividends" not in df.columns:
            df["dividends"] = 0.0

        if "stock_splits" not in df.columns:
            df["stock_splits"] = 0.0

        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values(["ticker", "date"]).drop_duplicates(
            subset=["ticker", "date"]
        )

        benchmark_prices.append(df)

    except Exception as e:
        print(f"  -> Failed for benchmark {ticker}: {e}")
        failed_benchmarks.append(ticker)

if not benchmark_prices:
    raise ValueError("No benchmark price data was downloaded.")

benchmark_prices = pd.concat(benchmark_prices, ignore_index=True)
benchmark_prices = benchmark_prices.sort_values(["ticker", "date"]).reset_index(drop=True)

benchmark_prices.to_parquet(BENCHMARK_PRICE_FILE, index=False)

print("\nBenchmark download complete.")
print(f"Saved benchmark prices to: {BENCHMARK_PRICE_FILE}")
print("Shape:", benchmark_prices.shape)
print("Failed benchmarks:", failed_benchmarks)

benchmark_prices.head()


Benchmark download complete.
Saved benchmark prices to: /Users/neilyejjey/stock_signal_engine_v1/data/raw/benchmark_prices.parquet
Shape: (7876, 10)
Failed benchmarks: []


,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits
0,2010-01-04,SPY,112.370003,113.389999,111.510002,113.330002,84.578468,118944600,0.0,0.0
1,2010-01-05,SPY,113.260002,113.680000,112.849998,113.629997,84.802353,111579900,0.0,0.0
2,2010-01-06,SPY,113.519997,113.989998,113.430000,113.709999,84.862068,116074400,0.0,0.0
3,2010-01-07,SPY,113.500000,114.330002,113.180000,114.190002,85.220284,131091100,0.0,0.0
4,2010-01-08,SPY,113.889999,114.620003,113.660004,114.570000,85.503876,126402800,0.0,0.0


In [9]:
# =========================================
# Download Current Snapshot Metadata
# =========================================
#
# WARNING:
# --------
# This is current metadata from yfinance.
# It must NOT be merged into historical training rows as if it were historical.
#
# Use cases:
#   - current deployment dashboard
#   - current market-cap sanity checks
#   - live scoring overlays
#   - portfolio explanation
#
# Not valid for:
#   - 2011-2022 historical model features

import numpy as np

CURRENT_SNAPSHOT_FILE = RAW_DIR / "current_ticker_snapshot.parquet"
CURRENT_SNAPSHOT_FAILED_FILE = RAW_DIR / "current_ticker_snapshot_failed.csv"

snapshot_rows = []
snapshot_failed = []

snapshot_fields = [
    "symbol",
    "shortName",
    "longName",
    "sector",
    "industry",
    "marketCap",
    "sharesOutstanding",
    "floatShares",
    "beta",
    "trailingPE",
    "forwardPE",
    "priceToBook",
    "enterpriseValue",
    "enterpriseToRevenue",
    "enterpriseToEbitda",
    "profitMargins",
    "operatingMargins",
    "grossMargins",
    "ebitdaMargins",
    "revenueGrowth",
    "earningsGrowth",
    "recommendationMean",
    "recommendationKey",
    "numberOfAnalystOpinions",
    "targetMeanPrice",
    "targetMedianPrice",
    "targetHighPrice",
    "targetLowPrice",
    "dividendYield",
    "payoutRatio",
    "currency",
    "exchange",
    "quoteType",
]

for i, ticker in enumerate(tickers, start=1):
    try:
        print(f"[{i}/{len(tickers)}] Fetching snapshot {ticker}...")

        info = yf.Ticker(ticker).get_info()

        if not info:
            snapshot_failed.append(ticker)
            continue

        row = {"ticker": ticker}

        for field in snapshot_fields:
            row[field] = info.get(field, np.nan)

        snapshot_rows.append(row)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        print(f"  -> Failed snapshot for {ticker}: {e}")
        snapshot_failed.append(ticker)

current_snapshot = pd.DataFrame(snapshot_rows)

# Standardize selected names
rename_map = {
    "marketCap": "current_market_cap",
    "sharesOutstanding": "current_shares_outstanding",
    "floatShares": "current_float_shares",
    "trailingPE": "current_trailing_pe",
    "forwardPE": "current_forward_pe",
    "priceToBook": "current_price_to_book",
    "enterpriseValue": "current_enterprise_value",
    "enterpriseToRevenue": "current_ev_to_revenue",
    "enterpriseToEbitda": "current_ev_to_ebitda",
    "profitMargins": "current_profit_margin",
    "operatingMargins": "current_operating_margin",
    "grossMargins": "current_gross_margin",
    "ebitdaMargins": "current_ebitda_margin",
    "revenueGrowth": "current_revenue_growth",
    "earningsGrowth": "current_earnings_growth",
    "recommendationMean": "current_recommendation_mean",
    "recommendationKey": "current_recommendation_key",
    "numberOfAnalystOpinions": "current_num_analyst_opinions",
    "targetMeanPrice": "current_target_mean_price",
    "targetMedianPrice": "current_target_median_price",
    "targetHighPrice": "current_target_high_price",
    "targetLowPrice": "current_target_low_price",
    "dividendYield": "current_dividend_yield",
    "payoutRatio": "current_payout_ratio",
}

current_snapshot = current_snapshot.rename(columns=rename_map)

current_snapshot["snapshot_date"] = pd.Timestamp.today().normalize()

current_snapshot.to_parquet(CURRENT_SNAPSHOT_FILE, index=False)

pd.DataFrame({"ticker": snapshot_failed}).to_csv(
    CURRENT_SNAPSHOT_FAILED_FILE,
    index=False,
)

print("\nCurrent snapshot download complete.")
print(f"Saved current snapshot to: {CURRENT_SNAPSHOT_FILE}")
print(f"Saved failed snapshot tickers to: {CURRENT_SNAPSHOT_FAILED_FILE}")
print("Shape:", current_snapshot.shape)
print("Failed:", len(snapshot_failed))

current_snapshot.head()

[1/503] Fetching snapshot MMM...
[2/503] Fetching snapshot AOS...
[3/503] Fetching snapshot ABT...
[4/503] Fetching snapshot ABBV...
[5/503] Fetching snapshot ACN...
[6/503] Fetching snapshot ADBE...
[7/503] Fetching snapshot AMD...
[8/503] Fetching snapshot AES...
[9/503] Fetching snapshot AFL...
[10/503] Fetching snapshot A...
[11/503] Fetching snapshot APD...
[12/503] Fetching snapshot ABNB...
[13/503] Fetching snapshot AKAM...
[14/503] Fetching snapshot ALB...
[15/503] Fetching snapshot ARE...
[16/503] Fetching snapshot ALGN...
[17/503] Fetching snapshot ALLE...
[18/503] Fetching snapshot LNT...
[19/503] Fetching snapshot ALL...
[20/503] Fetching snapshot GOOGL...
[21/503] Fetching snapshot GOOG...
[22/503] Fetching snapshot MO...
[23/503] Fetching snapshot AMZN...
[24/503] Fetching snapshot AMCR...
[25/503] Fetching snapshot AEE...
[26/503] Fetching snapshot AEP...
[27/503] Fetching snapshot AXP...
[28/503] Fetching snapshot AIG...
[29/503] Fetching snapshot AMT...
[30/503] Fetchi

,ticker,symbol,shortName,longName,sector,industry,current_market_cap,current_shares_outstanding,current_float_shares,beta,...,current_target_mean_price,current_target_median_price,current_target_high_price,current_target_low_price,current_dividend_yield,current_payout_ratio,currency,exchange,quoteType,snapshot_date
0,MMM,MMM,3M Company,3M Company,Industrials,Conglomerates,83429900288,521567261,5.206806e+08,1.095,...,170.78764,175.0,209.0,120.0,1.93,0.5723,USD,NYQ,EQUITY,2026-07-01
1,AOS,AOS,A.O. Smith Corporation,A. O. Smith Corporation,Industrials,Specialty Industrial Machinery,8564670464,111965465,1.097819e+08,1.175,...,70.45455,70.0,84.0,59.0,2.26,0.3733,USD,NYQ,EQUITY,2026-07-01
2,ABT,ABT,Abbott Laboratories,Abbott Laboratories,Healthcare,Medical Devices,160560332800,1741813154,1.731153e+09,0.620,...,116.54167,115.0,135.0,92.0,2.78,0.6723,USD,NYQ,EQUITY,2026-07-01
3,ABBV,ABBV,AbbVie Inc.,AbbVie Inc.,Healthcare,Drug Manufacturers - General,443571011584,1766792821,1.763648e+09,0.309,...,254.37930,258.0,328.0,200.0,2.75,3.2598,USD,NYQ,EQUITY,2026-07-01
4,ACN,ACN,Accenture plc,Accenture plc,Technology,Information Technology Services,80243965952,611942109,6.107855e+08,1.069,...,179.29080,179.0,275.0,130.0,5.24,0.5088,USD,NYQ,EQUITY,2026-07-01


In [10]:
# =========================================
# Data Inventory
# =========================================

data_inventory = []

for path in [
    TICKER_FILE,
    UNIVERSE_METADATA_FILE,
    PRICE_FILE,
    FAILED_FILE,
    BENCHMARK_PRICE_FILE,
    CURRENT_SNAPSHOT_FILE,
    CURRENT_SNAPSHOT_FAILED_FILE,
]:
    data_inventory.append({
        "file": str(path),
        "exists": path.exists(),
        "size_mb": path.stat().st_size / 1_000_000 if path.exists() else np.nan,
    })

data_inventory = pd.DataFrame(data_inventory)

data_inventory

,file,exists,size_mb
0,/Users/neilyejjey/stock_signal_engine_v1/data/...,True,0.002118
1,/Users/neilyejjey/stock_signal_engine_v1/data/...,True,0.053638
2,/Users/neilyejjey/stock_signal_engine_v1/data/...,True,49.197728
3,/Users/neilyejjey/stock_signal_engine_v1/data/...,True,0.000017
4,/Users/neilyejjey/stock_signal_engine_v1/data/...,True,0.372376
5,/Users/neilyejjey/stock_signal_engine_v1/data/...,True,0.125770
6,/Users/neilyejjey/stock_signal_engine_v1/data/...,True,0.000007
